# Importing,Data Loading and Overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

In [ ]:
df = pd.read_csv("C:/Users/anuj1/Downloads/us car/car_insurance_claim.csv")

In [ ]:
df.shape

In [ ]:
df.sample(5)

In [ ]:
df.columns.tolist()

In [ ]:
from prettytable import PrettyTable

# Create table object
table = PrettyTable()
table.field_names = ["Variable", "Explanation"]

# Add rows
rows = [
    ["ID", "Unique identifier for each customer."],
    ["KIDSDRIV", "Number of kids the customer has who are of driving age."],
    ["BIRTH", "Date of birth of the customer."],
    ["AGE", "Age of the customer."],
    ["HOMEKIDS", "Number of kids the customer has at home."],
    ["YOJ", "Years on the job (number of years the customer has been employed)."],
    ["INCOME", "Income of the customer."],
    ["PARENT1", "Indicates whether the customer is a single parent (Yes/No)."],
    ["HOME_VAL", "Value of the customer's home."],
    ["MSTATUS", "Marital status of the customer (Yes/No)."],
    ["GENDER", "Gender of the customer (M/F)."],
    ["EDUCATION", "Education level of the customer."],
    ["OCCUPATION", "Occupation of the customer."],
    ["TRAVTIME", "Daily travel time to work."],
    ["CAR_USE", "Purpose of car usage (Private/Commercial)."],
    ["BLUEBOOK", "Value of the customer's car."],
    ["TIF", "Time in force (Years)."],
    ["CAR_TYPE", "Type of car."],
    ["RED_CAR", "Indicates whether the customer owns a red car (yes/no)."],
    ["OLDCLAIM", "Total amount of previous claims."],
    ["CLM_FREQ", "Number of claims reported."],
    ["REVOKED", "Indicates whether the customer's driver's license has been revoked (Yes/No)."],
    ["MVR_PTS", "Number of motor vehicle record points."],
    ["CLM_AMT", "Possible future claims."],
    ["CAR_AGE", "Age of the customer's car."],
    ["CLAIM_FLAG", "Indicates whether a claim was filed (1/0)."],
    ["URBANICITY", "Urbanicity type ((Highly Urban/Urban)/(Highly Rural/Rural))."]
]

for row in rows:
    table.add_row(row)

# Print table
print(table)


In [ ]:
df.info()

# Missing value analysis
Purpose: Missing values can affect model performance and may need to be handled.

In [ ]:
df.isnull().sum()

In [ ]:
df['AGE'].fillna(df['AGE'].median(), inplace=True)

age column has a only 7 missing values, so filling with median

In [ ]:
df['YOJ'].fillna(df['YOJ'].median(), inplace=True)

year of joining column has approx 5% of data missing so filling with median value

In [ ]:
df['INCOME'] = df['INCOME'].replace('[\$,]', '', regex=True).astype(float)
df['HOME_VAL'] = df['HOME_VAL'].replace('[\$,]', '', regex=True).astype(float)

df['INCOME'].fillna(df['INCOME'].median(), inplace=True)
df['HOME_VAL'].fillna(df['HOME_VAL'].median(), inplace=True)

income and home value columns had object datatype so converted into float type and both column had approx 5% of data missing so i filled with meadian value.

In [ ]:
df['OCCUPATION'].fillna("Unknown", inplace=True)

occupation is categorical variable so filling with unknown.
mode can not be used because there are many categories and few have less difference among them.


In [ ]:
df['CAR_AGE'].fillna(df['CAR_AGE'].median(), inplace=True)

car_age column also have approx 6% of missing values so filled with meadian

In [ ]:
df['CAR_AGE'].median()

In [ ]:
df = df.replace({'z_': '', '<': ''}, regex=True)

In [ ]:
# Convert columns with dollar values to numeric
df['BLUEBOOK'] = df['BLUEBOOK'].replace('[\$,]', '', regex=True).astype(float)
df['OLDCLAIM'] = df['OLDCLAIM'].replace('[\$,]', '', regex=True).astype(float)
df['CLM_AMT'] = df['CLM_AMT'].replace('[\$,]', '', regex=True).astype(float)

In [ ]:
df.isnull().sum()

As you can see now after imputing values we have 0 missing values in our dataset.

In [ ]:
# Drop rows with missing claim amounts (target variable)
df = df.dropna(subset=['CLM_AMT'])
print("Remaining rows after cleaning:", len(df))

In [ ]:
df.describe()

# Univariate Analysis
Visualizing Numerical Features
We need to understand the distribution of important numerical variables

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from scipy.stats import gaussian_kde

num_cols = ['AGE', 'YOJ', 'INCOME', 'HOME_VAL', 'BLUEBOOK', 'CAR_AGE', 'MVR_PTS', 'CLM_AMT']

# Create subplots
fig = make_subplots(rows=3, cols=3, 
                   subplot_titles=num_cols,
                   vertical_spacing=0.1,
                   horizontal_spacing=0.05)

# Add histograms with KDE
for i, col in enumerate(num_cols):
    row = (i // 3) + 1
    col_pos = (i % 3) + 1
    
    # Create histogram
    fig.add_trace(
        go.Histogram(
            x=df[col],
            nbinsx=30,
            name=col,
            marker_color='#1f77b4',
            opacity=0.75,
            histnorm='probability density'  # For better KDE comparison
        ),
        row=row, col=col_pos
    )
    
    # Add KDE line
    data = df[col].dropna()
    if len(data) > 1:  # KDE requires at least 2 data points
        kde = gaussian_kde(data)
        x_range = np.linspace(data.min(), data.max(), 1000)
        fig.add_trace(
            go.Scatter(
                x=x_range,
                y=kde(x_range),
                mode='lines',
                line=dict(color='red', width=2),
                name='KDE'
            ),
            row=row, col=col_pos
        )

# Update layout
fig.update_layout(
    title_text='Distribution of Numerical Features with KDE',
    height=900,
    width=1200,
    showlegend=False,
    template='plotly_white'
)

# Update subplot titles
for i in range(len(num_cols)):
    fig.layout.annotations[i].update(font_size=12)

fig.show()

1.Distribution of age is approximately normal, centered around middle age(around 40)
2.Year of Job distribution is skews slightly left, indicating most data points are around 11 years or less.
3.The income distribution is right skewed, with most people havinga lower income of around 55k dollar , and 100k dollar for around 500 people and few with the higher income.
4. Home value this dist is heavily right-skewed with many having lower home values around 168k
5. Bluebook this dist is also right skewed representing the value of the customer's car with higher value of 12k doller.
6.Car age most car are relatively new, with the dist skewing right
7.Motor vehicle record points: Most individuals have a low count of points.
8.Claim amount: The distribution is jeavily skewed right, indicating most claims are low, with a few large claims.

In [ ]:
# List of numerical columns
num_cols = ['AGE', 'YOJ', 'INCOME', 'HOME_VAL', 'BLUEBOOK', 'CAR_AGE', 'MVR_PTS', 'CLM_AMT']

# Create histograms for numerical columns
plt.figure(figsize=(12, 8))
for i, col in enumerate(num_cols):
    plt.subplot(3, 3, i+1)
    sns.histplot(df[col], bins=30, kde=True)
    plt.title(f"Distribution of {col}")
plt.tight_layout()
plt.show()


# Insights:
#Skewness: if distributions are right-skewed (e.g., CLM_AMT), we need log transformation.
#Outliers: Large spikes in values indicate potential outliers.

1.Distribution of age is approximately normal, centered around middle age(around 40) 2.Year of Job distribution is skews slightly left, indicating most data points are around 11 years or less. 3.The income distribution is right skewed, with most people havinga lower income of around 55k dollar , and 100k dollar for around 500 people and few with the higher income. 4. Home value this dist is heavily right-skewed with many having lower home values around 168k 5. Bluebook this dist is also right skewed representing the value of the customer's car with higher value of 12k doller. 6.Car age most car are relatively new, with the dist skewing right 7.Motor vehicle record points: Most individuals have a low count of points. 8.Claim amount: The distribution is jeavily skewed right, indicating most claims are low, with a few large claims

In [ ]:
#Analyzing Categorical Features(Univariate analysis)
#We check how categorical variables are distributed.
# List of categorical columns
cat_cols = ['GENDER', 'CAR_USE', 'CAR_TYPE', 'PARENT1', 'MSTATUS', 'URBANICITY']

# Countplots for categorical columns
plt.figure(figsize=(12, 8))
for i, col in enumerate(cat_cols):
    plt.subplot(2, 3, i+1)
    sns.countplot(data=df, x=col, palette="coolwarm")
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# Insights:
#Which categories are most common?
#Imbalance in classes? (e.g., too many private vs. commercial cars).

This various bar charts representing distributions of different categories. 1.In terms of gender, there are slightly more individuals identified as Female than Male. 2.Most cars are used privately rather than commercially. 3.Minivans and SUV are the most common car types, while panel trucks and Van are the least common. 4.The majority of individuals do not identify as single parents . 5.A higher number of individuals are married versus single. Lastly, most individuals live in urban areas compared to rural ones.

# Bivariate Analysis (Relationships Between Variables)

In [ ]:
#Correlation Between Numeric Features

# Compute correlation matrix without the ID column
corr_matrix = df.drop(columns=['ID']).corr()

# Plot the heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Matrix (Without ID)")
plt.show()

 #Insights:
#High correlation between INCOME and HOME_VAL?
#MVR_PTS (driving violations) correlated with higher claim amounts?

In [ ]:
import plotly.figure_factory as ff
import numpy as np

# Compute correlation matrix without the ID column
corr_matrix = df.drop(columns=['ID']).corr()

# Convert correlation matrix to a NumPy array (rounded for better readability)
corr_values = np.round(corr_matrix.values, 2)

# Get feature names
features = corr_matrix.columns.tolist()

# Create an interactive heatmap using a supported colorscale
fig = ff.create_annotated_heatmap(z=corr_values, 
                                  x=features, 
                                  y=features, 
                                  colorscale="RdBu",  # Use a valid colorscale (e.g., "RdBu", "Viridis", "Blues")
                                  annotation_text=corr_values, 
                                  showscale=True)

# Update layout
fig.update_layout(title="Feature Correlation Matrix (Without ID)", 
                  xaxis=dict(tickangle=45), 
                  height=800, width=900, 
                  template="plotly_white")

# Show plot
fig.show()


Summary
The image is a feature correlation matrix displaying the relationships between different variables without including an ID. Correlations range from -1 to 1, with blue indicating positive correlations and orange/red indicating negative correlations.

Key observations:

KIDSDRIV has a strong positive correlation with HOMEKIDS (0.46) and a perfect negative correlation with itself (1.0).
CAR_AGE shows a moderate positive relationship with INCOME (0.39).
HOME_VAL and INCOME have a moderate positive correlation (0.55).
CLM_AMT and MVR_PTS are moderately positively correlated (0.08).
The strongest correlation is between each variable and itself (1.0 along the diagonal).
This matrix helps identify which variables may be more influential or related, whether positively or negatively, for further data analyses or modeling.

In [ ]:
from scipy.stats.mstats import winsorize

# Winsorize claim amount (caps extreme values at 5th and 95th percentile)
df['WINSOR_CLM_AMT'] = winsorize(df['CLM_AMT'], limits=[0.05, 0.05])

plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="CAR_TYPE", y="WINSOR_CLM_AMT", palette="coolwarm")
plt.xticks(rotation=45)
plt.title("Winsorized Claim Amounts by Car Type")
plt.show()

# Why?
#Some customers might have extremely high claim amounts (potential fraud?).
#If outliers exist, we cap extreme values using winsorization.

In [ ]:
import plotly.express as px

fig = px.box(df, x="CAR_TYPE", y="CLM_AMT", color="CAR_TYPE",
             title="Claim Amounts by Car Type (Interactive)", 
             hover_data=["CLM_AMT"], points="outliers")

fig.update_layout(yaxis=dict(title="Claim Amount", range=[0, df['CLM_AMT'].quantile(0.95)]))  # Limit the range
fig.show()


Key Takeaways:

Minivans, SUVs, and Pickups show high variance in claim amounts, with many high-value claims.

Sports Cars tend to have high claim amounts, possibly due to expensive repairs.

Vans & Panel Trucks generally have lower claim amounts, possibly indicating that they are less prone to severe damage or expensive repairs.

The outliers suggest that some vehicles experience extremely high claim amounts, likely due to severe accidents or expensive damages.

In [ ]:
#Pairplot (Multivariate Relationships)
#Shows relationships between numerical features.
#Helps detect clusters or trends.


# Selecting key numerical variables for better visualization
selected_features = ["AGE", "INCOME", "BLUEBOOK", "CLM_AMT", "MVR_PTS"]

# Pairplot
sns.pairplot(df[selected_features + ['CAR_USE']], diag_kind="kde", hue="CAR_USE")
plt.show()




#Are high-income people filing more claims?
#Do expensive cars (BLUEBOOK) have higher claim amounts?
#Do customers with high MVR_PTS have more severe claims?


In [ ]:
import plotly.express as px

# Selecting key numerical variables for better visualization
selected_features = ["AGE", "INCOME", "BLUEBOOK", "CLM_AMT", "MVR_PTS", "CAR_USE"]

# Convert categorical `CAR_USE` to string (Plotly needs categorical hue)
df['CAR_USE'] = df['CAR_USE'].astype(str)

# Create interactive scatter matrix
fig = px.scatter_matrix(df, 
                        dimensions=["AGE", "INCOME", "BLUEBOOK", "CLM_AMT", "MVR_PTS"], 
                        color="CAR_USE",
                        title="Pairwise Scatter Plot of Numerical Features",
                        labels={"CAR_USE": "Car Use"},
                        height=900)

# Show plot
fig.show()


Relationship Between Variables:

Income vs. Bluebook: There is a positive correlation, meaning higher car values tend to be associated with higher incomes.

Claim Amount vs. Bluebook: No strong correlation, implying car value does not necessarily determine claim amounts.

Claim Amount vs. MVR Points: Higher MVR Points (more violations) seem to be associated with slightly higher claim amounts.

In [ ]:
import numpy as np

df["LOG_CLM_AMT"] = np.log1p(df["CLM_AMT"])  # log(1 + CLM_AMT) to avoid log(0) issues

plt.figure(figsize=(10, 6))
sns.violinplot(data=df, x="CAR_TYPE", y="LOG_CLM_AMT", hue="CAR_USE", split=True, palette="coolwarm")
plt.title("Log-Transformed Claim Amount Distribution by Car Type")
plt.xticks(rotation=45)
plt.show()


Comparison Between Private vs. Commercial Vehicles
Commercial vehicles (orange) tend to have higher claim amounts on average compared to private vehicles.

The upper portion of the violin plots shows that commercial vehicles have a wider range of claim amounts.

Business Implications
Insurance companies should be cautious about commercial vehicles, as they have a higher risk of large claims.

Certain vehicle types (like SUVs & Vans) have a wider spread, suggesting more unpredictable claim amounts.

Pricing strategies for insurance premiums could be adjusted based on vehicle type and usage.



In [ ]:
#KDE Plot (Kernel Density Estimation)
#Compares distributions of numerical features (useful for fraud detection).
plt.figure(figsize=(10, 5))
sns.kdeplot(df[df['CLM_AMT'] > 0]['INCOME'], fill=True, label="Filed Claims", color="red")
sns.kdeplot(df[df['CLM_AMT'] == 0]['INCOME'], fill=True, label="No Claims", color="blue")
plt.title("Income Distribution: Claim vs No Claim")
plt.legend()
plt.show()


#Insights:
#Do lower-income groups file more claims?
#Helps in fraud detection (unexpectedly high claims in low-income groups).

In [ ]:
import plotly.figure_factory as ff
import numpy as np

# Extract income values for both groups
income_filed = df[df['CLM_AMT'] > 0]['INCOME'].dropna()
income_not_filed = df[df['CLM_AMT'] == 0]['INCOME'].dropna()

# Create KDE plot using Plotly
fig = ff.create_distplot([income_filed, income_not_filed], 
                         group_labels=["Filed Claims", "No Claims"], 
                         colors=["red", "blue"], 
                         show_hist=False, show_rug=True)

# Update layout
fig.update_layout(title="Income Distribution: Claim vs No Claim",
                  xaxis_title="Income",
                  yaxis_title="Density",
                  template="plotly_white")

# Show plot
fig.show()



Insights:
The data suggests that lower-income individuals tend to file more claims compared to higher-income groups.

As income increases, the likelihood of filing a claim decreases.

This insight can be useful for insurance companies to design better risk-based pricing strategies.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

# Load your dataset (replace with actual file path if needed)
# df = pd.read_csv("your_data.csv")

# Apply log transformation to claim amounts
df["LOG_CLM_AMT"] = np.log1p(df["CLM_AMT"])  # log(1 + x) transformation

# Create scatter plot using Plotly
fig = px.scatter(
    df, 
    x="CAR_TYPE", 
    y="LOG_CLM_AMT", 
    color="REVOKED", 
    title="Log Transformed Claim Amounts by Car Type and Revoked License Status",
    labels={"LOG_CLM_AMT": "Log(Claim Amount)", "CAR_TYPE": "Car Type"},
    opacity=0.7
)

# Show the interactive plot
fig.show()


Insights:

Across all car types, most claims cluster between log(Claim Amount) ≈ 7-10, which translates to approx. ₹1,000 to 22,000 in actual values.

Car Type Comparisons
SUVs, Vans, and Pickup trucks seem to have higher variability in claim amounts.

Minivans and Sports Cars have slightly more compact distributions, indicating less variation in claims.

Panel Trucks show fewer high-value claims, suggesting they might have lower insurance risks.

In [ ]:
#Detects hidden relationships between categorical variables.
plt.figure(figsize=(12, 6))
sns.heatmap(pd.crosstab(df['CAR_TYPE'], df['URBANICITY']), annot=True, cmap="coolwarm", fmt='d')
plt.title("Heatmap of Car Type vs Urbanicity")
plt.show()

 #Insights:
#Which car types are more common in urban vs rural areas?
#Helps adjust risk pricing based on location.

Insights:
Charge Higher Premiums for SUVs and Minivans in Urban Areas

SUVs (2228) and Minivans (2184) are the most common car types in urban areas.

Urban areas have higher accident rates, theft risks, and repair costs due to traffic congestion and population density.

Action: Increase premium rates for these vehicles in urban locations to account for higher risk exposure.

Rural customers have fewer vehicle choices (lower counts of Panel Trucks, Vans, and Sports Cars).

Action: Offer pay-as-you-drive or low-mileage discounts for rural customers, as they drive less and pose lower risk.

In [ ]:
#Cluster Analysis (KMeans for Segmentation)
#Groups customers into risk-based clusters.
from sklearn.cluster import KMeans

# Select relevant features
X_cluster = df[['AGE', 'INCOME', 'BLUEBOOK', 'MVR_PTS']]
X_cluster = X_cluster.dropna()

# Train KMeans Model
kmeans = KMeans(n_clusters=3, random_state=42)
df['Risk_Cluster'] = kmeans.fit_predict(X_cluster)

# Plot Clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="AGE", y="CLM_AMT", hue="Risk_Cluster", palette="coolwarm")
plt.title("Customer Clusters: Age vs Claim Amount")
plt.show()


 #Insights:
#Helps segment customers into Low, Medium, and High-Risk Groups.
#Insurance companies use risk clustering for pricing strategies.



In [ ]:
# Select features for clustering
cluster_features = ["AGE", "INCOME", "BLUEBOOK", "MVR_PTS", "CLM_FREQ"]
X_cluster = df[cluster_features].dropna()

# Standardize features for better clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Train KMeans Model
kmeans = KMeans(n_clusters=3, random_state=42)
df['Risk_Cluster'] = kmeans.fit_predict(X_scaled)

# Plot Clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="AGE", y="CLM_AMT", hue="Risk_Cluster", palette="coolwarm")
plt.title("Enhanced Customer Clustering: Age vs Claim Amount")
plt.show()


#Insights:
Risk & Claim Trends:

Middle-aged customers (30-50 years old) have the highest claim amounts and the highest number of high-risk customers.

Younger customers (under 30) generally have lower claims and are mostly low-risk.

Older customers (50+) show moderate claim amounts, with a mix of medium and high-risk individuals.


In [ ]:
#Log Transformation for Skewed Data

# Apply log transformation to highly skewed columns
df['LOG_CLM_AMT'] = np.log1p(df['CLM_AMT'])
df['LOG_INCOME'] = np.log1p(df['INCOME'])
df['LOG_BLUEBOOK'] = np.log1p(df['BLUEBOOK'])

# Plot log-transformed claim amount
sns.histplot(df['LOG_CLM_AMT'], bins=30, kde=True)
plt.title("Log-Transformed Claim Amount Distribution")
plt.show()


#Insights:

A significant portion of the data is concentrated at LOG_CLM_AMT = 0, indicating many customers have zero claims.

The remaining distribution is right-skewed, suggesting that among those who file claims, the amounts vary significantly, but most are on the lower side.

The large number of zero-claim customers suggests that many policyholders may be low-risk, opening opportunities for targeted discounts or lower premiums.

The right-skewed claim distribution indicates that a small percentage of customers file high claims, meaning insurers should focus on risk assessment and fraud detection for high-claim segments.


In [ ]:
# Check basic statistics
print(df['AGE'].describe())

# Histogram to check distribution
plt.figure(figsize=(6,4))
sns.histplot(df['AGE'], bins=30, kde=True)
plt.title("Age Distribution")
plt.show()

# Boxplot to check for outliers
plt.figure(figsize=(4,3))
sns.boxplot(x=df['AGE'])
plt.title("Age Outliers")
plt.show()

# Binning Example (Creating Age Groups)
df['AGE_GROUP'] = pd.cut(df['AGE'], bins=[0, 25, 40, 60, 100], labels=['Young', 'Middle', 'Senior', 'Elder'])
print(df['AGE_GROUP'].value_counts())


Age Distribution Insights
Demographic Concentration

The age distribution is approximately normal, with a peak around 45 years.

Most individuals fall between 39 and 51 years (IQR), while the mean age is 44.84 years.

The majority of customers belong to the "Senior" (6772) and "Middle" (3083) age groups.

Outliers in Age

Outliers are observed at both extremes (below 20 and above 70 years).

These could represent exceptional cases, such as early or late insurance claimants.

In [ ]:
# Check basic statistics
print(df['INCOME'].describe())

# Check missing values
print(f"Missing INCOME values: {df['INCOME'].isnull().sum()}")

# Histogram for distribution
plt.figure(figsize=(6,4))
sns.histplot(df['INCOME'], bins=30, kde=True)
plt.title("Income Distribution")
plt.show()

# Boxplot for outliers
plt.figure(figsize=(4,3))
sns.boxplot(x=df['INCOME'])
plt.title("Income Outliers")
plt.show()

# Log transformation if right-skewed
df['LOG_INCOME'] = np.log1p(df['INCOME'])

# Check correlation with CLM_AMT (Claim Amount)
print(df[['INCOME', 'CLM_AMT']].corr())
sns.scatterplot(x=df['INCOME'], y=df['CLM_AMT'])
plt.title("Income vs Claim Amount")
plt.show()


Income Distribution Overview
The average income is ₹61,127, with a median of ₹53,529, indicating a slightly right-skewed distribution.

25% of customers earn below ₹29,165, while the top 25% earn above ₹83,231, showing high income disparity.

The maximum income reaches ₹3,67,030, which is significantly higher than the 75th percentile, suggesting the presence of high-income outliers.

 Correlation Between Income and Claim Amount (CLM_AMT)
Very weak negative correlation (-0.0559) between income and claim amount, meaning income does not significantly impact claim amount.

This suggests that higher-income customers do not necessarily file higher claims, possibly due to better financial planning or different insurance coverage choices.

In [ ]:
df['LOG_INCOME'] = np.log1p(df['INCOME'])  # log(1 + income) to avoid log(0)
sns.histplot(df['LOG_INCOME'], bins=30, kde=True)
plt.title("Log Transformed Income Distribution")
plt.show()


Mode (Peak) of log-income:

The peak appears around log-income values between 10 and 11, suggesting that the most common income levels, after log transformation, fall in this range.

So most individuals are likely earning $22K to $60K annually.

Zero or Very Low Incomes: A sharp spike at log-income = 0 suggests a significant number of individuals with zero or very low reported income.

In [ ]:
df['INCOME_GROUP'] = pd.cut(df['INCOME'], 
                            bins=[0, 30000, 70000, 150000], 
                            labels=['Low', 'Medium', 'High'])


print(df['INCOME_GROUP'].value_counts())
sns.countplot(x=df['INCOME_GROUP'], order=['Low', 'Medium', 'High'])
plt.title("Income Groups Distribution")
plt.show()

Most People Have a Medium Income: The largest group (4,157 people) falls in the medium-income range, meaning their earnings are around the average.

High-Income Group is Significant: A good number (2,995 people) earn a high income, but they are fewer than those in the medium category.

Fewer People Have Low Income: Only 1,844 people fall into the low-income category, making them the smallest group.

In [ ]:
# Check unique values
print(df['OCCUPATION'].value_counts())

# Barplot of categories
plt.figure(figsize=(8,4))
sns.countplot(y=df['OCCUPATION'], order=df['OCCUPATION'].value_counts().index)
plt.title("Occupation Distribution")
plt.show()

# Check for missing values
print(f"Missing OCCUPATION values: {df['OCCUPATION'].isnull().sum()}")

# If too many categories, group similar ones
df['OCCUPATION_GROUPED'] = df['OCCUPATION'].replace({
    'Doctor': 'Healthcare',
    'Nurse': 'Healthcare',
    'Software Engineer': 'IT',
    'Data Scientist': 'IT',
    'Teacher': 'Education',
    # Add more groupings as needed
})

# One-hot encoding if categorical encoding is needed
df = pd.get_dummies(df, columns=['OCCUPATION_GROUPED'], drop_first=True)


Most Common Occupation: Blue-collar jobs (2,288 people) dominate the dataset.

Office and Skilled Jobs: Clerical (1,590), professional (1,408), and managerial (1,257) roles are significant.

Specialized Professions: Lawyers (1,031) and doctors (321) represent high-skilled fields.

Students and Home Makers: A notable portion includes students (899) and home makers (843).

Unknown Category: 665 individuals have unspecified occupations, possibly retirees or unemployed.

In [ ]:
sns.lmplot(data=df, x='TIF', y='CLM_AMT', lowess=True, line_kws={'color': 'red'})
plt.title("Time in Force vs. Claim Amount")
plt.show()

Higher Claims Occur Early
Most high claim amounts are concentrated when TIF is between 0 and 5 years.
This suggests larger claims are more frequent early in a policy's life.

Claim Amount Decreases Over Time
As TIF increases, the claim amounts tend to decrease, forming a downward trend.
The red regression line confirms a negative relationship between TIF and claim amount.

Long-Term Policies Have Lower Claims
For policies active beyond 10 years, claim amounts are generally much lower and less frequent.

In [ ]:
fig = px.violin(df, x="REVOKED", y="CLM_AMT", box=True, points="all", 
                title="Claim Severity by License Revocation Status")
fig.show()

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
# Winsorizing at 5th and 95th percentiles
lower = df['CLM_AMT'].quantile(0.05)
upper = df['CLM_AMT'].quantile(0.95)

df['CLM_AMT_WINSORIZED'] = np.clip(df['CLM_AMT'], lower, upper)

# Normalize
scaler = MinMaxScaler()
df['CLM_AMT_NORMALIZED'] = scaler.fit_transform(df[['CLM_AMT_WINSORIZED']])

# Plot
fig = px.violin(df,
                x='REVOKED',
                y='CLM_AMT_NORMALIZED',
                points='all',
                box=True,
                color='REVOKED',
                title='Normalized Claim Severity (Winsorized) by License Revocation Status')

fig.update_layout(yaxis_title='Normalized CLM_AMT')
fig.show()


Distribution Shape Difference:

Revoked = Yes group shows a wider spread of normalized claim amounts, especially in the mid to upper range.

Revoked = No group is more tightly clustered near the lower end, indicating more consistent and smaller claim amounts.

In [ ]:
pivot = df.pivot_table(values='CLM_AMT', index='URBANICITY', columns='CAR_USE', aggfunc='mean')
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="Blues")
plt.title("Average Claim Amount by Urbanicity and Car Use")
plt.show()

Claim Amounts Are Higher in Urban Areas:
Both commercial and private vehicle claims are significantly higher in highly urban/urban areas compared to rural areas.

For example:
Commercial (Urban): 2555
Commercial (Rural): 427
That’s almost a 6x increase in urban regions.

Especially in urban areas, commercial vehicles have the highest average claims (2555), likely due to more road exposure and risk

In [ ]:

import plotly.express as px

fig = px.treemap(df, path=['OCCUPATION'], values='CLM_AMT',
                 title="Claim Amount by Occupation")

# Show the values (claim amounts) as labels inside the treemap
fig.update_traces(textinfo='label+value')  # label = OCCUPATION, value = CLM_AMT

fig.show()


Top Occupation – Blue Collar
Blue Collar workers have the highest total claim amount at 4,752,674.
This indicates they are either involved in more accidents or more expensive ones, possibly due to the nature of their jobs (e.g., physical labor, higher road exposure).

Other High Claim Categories
Clerical (2.57M), Professional (2.1M), and Student (1.77M) occupations also show significant total claim amounts.
Clerical roles might involve regular commuting; students may be inexperienced drivers.

Lowest Claim Amount – Doctor
Doctors have the lowest claim total, suggesting fewer claims or lower amounts per claim, possibly due to lower driving frequency or more cautious driving behavior.

In [ ]:
# T-test for red vs. non-red cars
from scipy.stats import ttest_ind
red_claims = df[df['RED_CAR'] == 'yes']['CLM_AMT']
nonred_claims = df[df['RED_CAR'] == 'no']['CLM_AMT']
t_stat, p_value = ttest_ind(red_claims, nonred_claims)
print(f"Red cars vs. Others: p-value = {p_value:.4f}")

# Boxplot
plt.figure(figsize=(8,4))
sns.boxplot(data=df, x='RED_CAR', y='CLM_AMT')
plt.title("Claim Amount by Red Car Ownership")
plt.show()

In [ ]:
# Barplot (Plotly)
fig = px.bar(df, x='EDUCATION', y='CLM_AMT', color='EDUCATION', 
             title="Average Claim Amount by Education Level")
fig.show()

Inverse Relationship Between Education and Claim Amounts:
Generally, as education level increases, claim amounts tend to decrease.

Potential Business Insight for Insurers:

Individuals with only a high school education might represent a higher insurance risk segment compared to those with advanced degrees.

In [ ]:
# Scatterplot with regression
sns.lmplot(data=df, x='TRAVTIME', y='CLM_AMT', hue='CAR_USE', 
           line_kws={'color': 'black'}, height=6)
plt.title("Travel Time vs. Claim Amount")
plt.show()

Commercial use may be a risk factor for higher claim amounts.

Travel time is not a strong predictor of claim amount on its own.

Most incidents (and claims) occur during shorter trips, which may be due to higher trip frequency or urban driving conditions.

Outliers (very high claims) are more often associated with commercial use, which could inform premium pricing models.

In [ ]:
#Why: Teen drivers significantly increase accident risk

df['LOG_CLM_AMT'] = np.log1p(df['CLM_AMT'])  # Apply log transformation
px.box(df, x='KIDSDRIV', y='LOG_CLM_AMT', color='CAR_USE', 
       title="Log-Transformed Claim Amount by Teen Drivers & Vehicle Use")


Teen drivers don’t always mean higher claims – Having more teen drivers doesn't consistently raise or lower the claim amount. It stays fairly similar across different numbers.

Commercial cars often have higher claims – On average, commercial vehicles tend to show higher claim amounts than private ones, especially with 1–2 teen drivers.

More variation with private use – Private cars with many teen drivers (like 3 or 4) show more ups and downs in claims, meaning the claim amounts vary a lot.

In [ ]:
df.info()

In [ ]:
df.shape

# Feature Creation

Feature engineering plays a critical role in improving the predictive performance of machine learning models. In this study, we applied various feature transformations, new feature creations, and selection techniques to enhance data quality and reduce redundancy. The key steps taken are detailed below.


In [ ]:
print(df['GENDER'].unique())        # See unique values in GENDER column
print(df['URBANICITY'].unique())    # See unique values in URBANICITY column
print(df['CAR_TYPE'].unique())
print(df['OCCUPATION'].unique())

In [ ]:
df['GENDER'].info()
df['URBANICITY'].info()
df['CAR_TYPE'].info()
df['OCCUPATION'].info()

In [ ]:
df['GENDER'] = df['GENDER'].map({'M': 0, 'F': 1})
df['URBANICITY'] = df['URBANICITY'].map({'Highly Rural/ Rural': 0, 'Highly Urban/ Urban': 1})


In [ ]:
import pandas as pd
import numpy as np

# Assuming df is your dataframe
df['INCOME_GROUP'].fillna(df['INCOME_GROUP'].mode()[0], inplace=True)

# Create a missing indicator
df['INCOME_GROUP_MISSING'] = df['INCOME_GROUP'].isna().astype(int)


In [ ]:
# Binary encoding for categorical variables
binary_cols = ['PARENT1', 'MSTATUS',  'REVOKED']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})


In [ ]:
print(df['EDUCATION'].unique())



In [ ]:
df['EDUCATION'] = df['EDUCATION'].astype(str)


In [ ]:
education_order = [['Unknown', 'Below High School', 'High School', 'Bachelor', 'Master', 'PhD']]


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Standardize category names in EDUCATION
education_mapping = {
    'Bachelors': 'Bachelor',
    'Masters': 'Master',
    'Phd': 'PhD',  # In case 'Phd' is lowercase
    'phd': 'PhD',  # Fix lowercase issues
    'masters': 'Master',
    'bachelors': 'Bachelor'
}

df['EDUCATION'] = df['EDUCATION'].replace(education_mapping)

# Fill NaN values with 'Unknown'
df['EDUCATION'].fillna('Unknown', inplace=True)

# Define the correct order (matching the standardized categories)
education_order = [['Unknown', 'Below High School', 'High School', 'Bachelor', 'Master', 'PhD']]

# Initialize OrdinalEncoder
ordinal_encoder = OrdinalEncoder(categories=education_order)

# Apply encoding
df[['EDUCATION']] = ordinal_encoder.fit_transform(df[['EDUCATION']])

# Convert EDUCATION column to integers for better readability
df['EDUCATION'] = df['EDUCATION'].astype(int)

# Display the first few rows
print(df[['EDUCATION']].head())



In [ ]:
print(df['CAR_TYPE'].unique())


In [ ]:
df = pd.get_dummies(df, columns=['CAR_TYPE'], prefix='CT', drop_first=True)


In [ ]:
print(df['OCCUPATION'].unique()) 

In [ ]:
df = pd.get_dummies(df, columns=['OCCUPATION'], prefix=['OCC'], drop_first=True)


In [ ]:
df.info()

In [ ]:
# Claim Frequency per Time
df['CLAIM_RATE'] = df['CLM_FREQ'] / (df['TIF'] + 1)

# Risk Score based on violations
df['RISK_SCORE'] = df['MVR_PTS'] * df['CLM_FREQ']

# Claim per Car Age
df['CLAIM_AGE_RATIO'] = df['OLDCLAIM'] / (df['CAR_AGE'] + 1)

# Claim to Income Ratio
df['CLAIM_TO_INCOME_RATIO'] = df['CLM_AMT'] / (df['INCOME'] + 1)


In [ ]:
from scipy.stats.mstats import winsorize

# Winsorize claim amount
df['CLM_AMT_WINSORIZED'] = winsorize(df['CLM_AMT'], limits=[0.05, 0.05])


In [ ]:
df['LOG_CLAIM_RATE'] = np.log1p(df['CLAIM_RATE'])
df['LOG_INCOME'] = np.log1p(df['INCOME'])
df['LOG_BLUEBOOK'] = np.log1p(df['BLUEBOOK'])


In [ ]:
# Old Car Indicator
df['OLD_CAR'] = (df['CAR_AGE'] > 10).astype(int)

In [ ]:
print(df['BIRTH'].dtype)


In [ ]:

df['BIRTH'] = pd.to_datetime(df['BIRTH'], errors='coerce')


In [ ]:
# Driver experience calculation
current_year = pd.Timestamp.now().year
df['DRIVING_EXPERIENCE'] = current_year - df['BIRTH'].dt.year - 16

df['GENERATION'] = pd.cut(df['BIRTH'].dt.year,
                         bins=[1925, 1945, 1965, 1980, 1995, 2010],
                         labels=['Silent', 'Baby Boomer', 'Gen X', 'Millennial', 'Gen Z'])


In [ ]:
# Job stability index
df['JOB_STABILITY'] = df['YOJ'] / (df['AGE'] - 18 + 1)  

# Career interruption flag
df['CAREER_GAP'] = ((df['AGE'] - 18) > df['YOJ']).astype(int)

In [ ]:
# Family responsibility score
df['FAMILY_RISK'] = (df['PARENT1'] + 
                    df['HOMEKIDS'] * 0.5 + 
                    df['KIDSDRIV'] * 0.7)

# Teen driver risk
df['TEEN_DRIVER_RISK'] = df['KIDSDRIV'] * df['MVR_PTS']

FAMILY_RISK=PARENT1+(HOMEKIDS×0.5)+(KIDSDRIV×0.7)

Why?
Measures the financial and personal responsibility an individual has towards dependents.

Higher responsibility could impact financial risk and claim likelihood.

In [ ]:
# Select already existing one-hot encoded columns
occupation_cols = [col for col in df.columns if col.startswith('OCC_')]
car_type_cols = [col for col in df.columns if col.startswith('CT_')]

# Check if they exist
if occupation_cols and car_type_cols:
    print("Using one-hot encoded OCCUPATION and CAR_TYPE features")
else:
    print("Warning: Some occupation or car type columns might be missing!")


In [ ]:
df['URBANICITY'] = df['URBANICITY'].astype('category').cat.codes
df['INCOME_GROUP'] = df['INCOME_GROUP'].astype('category').cat.codes


In [ ]:
df['URBAN_WEALTH'] = df['URBANICITY'] * df['INCOME_GROUP']


In [ ]:
# Urban wealth indicator
df['URBAN_WEALTH'] = df['URBANICITY'] * df['INCOME_GROUP']

df['RISKY_VEHICLE'] = ((df[['CT_SUV', 'CT_Sports Car']].sum(axis=1) > 0) & 
                        (df['BLUEBOOK'] > df['BLUEBOOK'].median())).astype(int)


In [ ]:
# Claim acceleration rate
df['CLAIM_ACCELERATION'] = df['CLM_FREQ'] / df['TIF']

# Policy lifetime value
df['POLICY_LTV'] = df['TIF'] * df['INCOME'] / 1000

In [ ]:
from sklearn.preprocessing import PowerTransformer

# Apply Yeo-Johnson transformation
pt = PowerTransformer()
transform_features = ['INCOME', 'HOME_VAL', 'BLUEBOOK']
df[transform_features] = pt.fit_transform(df[transform_features])

In [ ]:
# Claim severity ratio
df['SEVERITY_RATIO'] = df['CLM_AMT_WINSORIZED'] / df['BLUEBOOK']

# Risk-adjusted premium
df['RISK_ADJ_PREMIUM'] = (df['MVR_PTS'] * 0.2 + 
                         df['CLM_FREQ'] * 0.3 + 
                         df['CAR_AGE'] * 0.1)

In [ ]:
# Urban density multiplier
df['URBAN_RISK_MULTIPLIER'] = np.where(
    df['URBANICITY'] == 1, 
    1.25 * df['TRAVTIME'], 
    0.9 * df['TRAVTIME']
)

# Rural claim penalty
df['RURAL_PENALTY'] = ((df['URBANICITY'] == 0) & 
                      (df['CAR_USE'] == 'Commercial')).astype(int) * 1.15

In [ ]:
df.columns.tolist()

In [ ]:
final_features = [
    'AGE', 'GENDER', 'EDUCATION', 'PARENT1', 'MSTATUS', 'INCOME',
    'HOME_VAL', 'BLUEBOOK', 'LOG_INCOME', 'LOG_BLUEBOOK', 'MVR_PTS',
    'REVOKED', 'TRAVTIME', 'CLAIM_RATE', 'LOG_CLAIM_RATE', 'CAR_AGE',
    'OLD_CAR', 'URBANICITY', 'CLM_FREQ', 'OLDCLAIM', 'CLAIM_AGE_RATIO',
    'RISK_SCORE', 'CLAIM_TO_INCOME_RATIO',
    
    # One-Hot Encoded CAR_TYPE Features
    'CT_Panel Truck', 'CT_Pickup', 'CT_SUV', 'CT_Sports Car', 'CT_Van',

    # One-Hot Encoded OCCUPATION Features
    'OCC_Clerical', 'OCC_Doctor', 'OCC_Home Maker', 'OCC_Lawyer', 'OCC_Manager',
    'OCC_Professional', 'OCC_Student', 'OCC_Unknown'
]

# Create the final dataset
df_final = df[final_features]

# Print final dataset shape
print(f" Final dataset created with shape: {df_final.shape}")


In [ ]:
# Ensure CLAIM_AMOUNT (CLM_AMT) is included in df_final
if 'CLM_AMT' in df.columns and 'CLAIM_AMOUNT' not in df_final.columns:
    df_final['CLAIM_AMOUNT'] = df['CLM_AMT']


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data["feature"] = df[final_features].columns
vif_data["VIF"] = [variance_inflation_factor(df[final_features].values, i) 
                   for i in range(len(df[final_features].columns))]

print("VIF Scores:")
print(vif_data.sort_values('VIF', ascending=False))


In [ ]:
# Drop highly collinear features
df_final.drop(['LOG_BLUEBOOK', 'LOG_CLAIM_RATE', 'CLAIM_RATE', 'LOG_INCOME'], axis=1, inplace=True)

# Recalculate VIF after dropping
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data["feature"] = df_final.columns
vif_data["VIF"] = [variance_inflation_factor(df_final.values, i) for i in range(len(df_final.columns))]
print(vif_data)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numerical_cols = ['AGE', 'INCOME', 'HOME_VAL', 'BLUEBOOK', 'MVR_PTS', 'TRAVTIME', 'CAR_AGE', 'RISK_SCORE']
df_final[numerical_cols] = scaler.fit_transform(df_final[numerical_cols])


# Model building

Model building is the process of selecting, training, and optimizing a machine learning algorithm to learn patterns from data and make predictions. It involves several key steps, including data preprocessing, feature selection, algorithm selection, hyperparameter tuning, model training, and evaluation. The goal is to develop a robust model that generalizes well to unseen data.

In [ ]:
from sklearn.model_selection import train_test_split

# Define X (features) and y (target)
X = df_final.drop(columns=['CLAIM_AMOUNT'])  # Remove target column from features
y = df_final['CLAIM_AMOUNT']  # Target variable

# Split data into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
print(df_final.columns)  # Check if 'CLAIM_AMOUNT' exists


In [ ]:
# Step 1: Import Libraries
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import SelectFromModel
import numpy as np

# Step 2: Define Features (X) and Target (y)
X = df_final.drop(columns=['CLAIM_AMOUNT'])  # Features
y = df_final['CLAIM_AMOUNT']  # Target variable

# Step 3: Split Data into Training & Testing Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Train XGBoost Regressor
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    random_state=42
)

# Fit the model (without early stopping to avoid compatibility issues)
xgb_model.fit(X_train, y_train)

# Step 5: Feature Selection
selector = SelectFromModel(estimator=xgb_model, threshold="median", prefit=True)
X_train_sel = selector.transform(X_train)
X_test_sel = selector.transform(X_test)

print(f"\nOriginal features: {X.shape[1]}")
print(f"Selected features: {X_train_sel.shape[1]}")

# Step 6: Retrain XGBoost on Selected Features
xgb_model_sel = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    random_state=42
)
xgb_model_sel.fit(X_train_sel, y_train)

# Step 7: Make Predictions
y_train_pred = xgb_model_sel.predict(X_train_sel)
y_test_pred = xgb_model_sel.predict(X_test_sel)

# Step 8: Evaluate Model Performance
print("\n🔹 Training Data:")
print(f"   MAE: {mean_absolute_error(y_train, y_train_pred):.2f}")
print(f"   RMSE: {mean_squared_error(y_train, y_train_pred) ** 0.5:.2f}")
print(f"   R²: {r2_score(y_train, y_train_pred):.4f}")

print("\n🔹 Test Data:")
print(f"   MAE: {mean_absolute_error(y_test, y_test_pred):.2f}")
print(f"   RMSE: {mean_squared_error(y_test, y_test_pred) ** 0.5:.2f}")
print(f"   R²: {r2_score(y_test, y_test_pred):.4f}")


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 1],
}

xgb_rs = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(random_state=42),
    param_distributions=param_grid,
    scoring='neg_mean_absolute_error',
    cv=7,
    n_iter=20,
    random_state=42,
    verbose=1,
    n_jobs=-1
)

xgb_rs.fit(X_train_sel, y_train)
print("Best parameters:", xgb_rs.best_params_)


In [ ]:
from sklearn.feature_selection import SelectFromModel

# Fit model
xgb_model.fit(X_train, y_train)

# Feature selection
selector = SelectFromModel(estimator=xgb_model, threshold="median", prefit=True)

# Transform data
X_selected = selector.transform(X)


In [ ]:
xgb_best = xgb.XGBRegressor(
    subsample=0.8,
    reg_alpha=1,
    n_estimators=100,
    max_depth=8,
    learning_rate=0.1,
    colsample_bytree=1.0,
    random_state=42
)

xgb_best.fit(X_train, y_train)


In [ ]:
#tuned xgboost model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_train_pred = xgb_best.predict(X_train)
y_test_pred = xgb_best.predict(X_test)

print("🔹 Train Performance")
print(f"MAE: {mean_absolute_error(y_train, y_train_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.2f}")
print(f"R²: {r2_score(y_train, y_train_pred):.4f}")

print("\n🔹 Test Performance")
print(f"MAE: {mean_absolute_error(y_test, y_test_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.2f}")
print(f"R²: {r2_score(y_test, y_test_pred):.4f}")


In [ ]:
import pickle

# Save the XGBoost tuned model
with open("xgboost_best.pkl", "wb") as f:
    pickle.dump(xgb_best, f)

print(" XGBoost model saved as 'xgboost_best.pkl'")


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Initialize model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# Train the model
rf_model.fit(X_train, y_train)


In [ ]:
# Get feature importances
importances = rf_model.feature_importances_
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# Plot feature importance
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance.head(15))
plt.title('Top 15 Feature Importances')
plt.show()

In [ ]:
# Step 1: Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

# Step 2: Prepare Data
# Replace df_final with your actual DataFrame if named differently
X = df_final.drop(columns=['CLAIM_AMOUNT'])
y = df_final['CLAIM_AMOUNT']

# Step 3: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Use the best parameters from GridSearchCV
best_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# Step 5: Train the model
best_rf.fit(X_train, y_train)

# Step 6: Predict
y_train_pred = best_rf.predict(X_train)
y_test_pred = best_rf.predict(X_test)

# Step 7: Evaluate
print("🔹 Train Performance:")
print(f"   MAE: {mean_absolute_error(y_train, y_train_pred):.2f}")
print(f"   RMSE: {mean_squared_error(y_train, y_train_pred)**0.5:.2f}")
print(f"   R²: {r2_score(y_train, y_train_pred):.4f}")

print("\n🔹 Test Performance:")
print(f"   MAE: {mean_absolute_error(y_test, y_test_pred):.2f}")
print(f"   RMSE: {mean_squared_error(y_test, y_test_pred)**0.5:.2f}")
print(f"   R²: {r2_score(y_test, y_test_pred):.4f}")

# Optional: Step 8 - Cross Validation
print("\n🔸 Cross-Validation (5-Fold) MAE:")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_mae_scores = cross_val_score(best_rf, X, y, cv=kf, scoring='neg_mean_absolute_error')
mae_scores = -cv_mae_scores
print(f"Scores: {mae_scores}")
print(f"Average MAE: {mae_scores.mean():.2f}")

# Optional: Step 9 - Plot CV MAE per Fold
plt.plot(range(1, 6), mae_scores, marker='o')
plt.title('Cross-Validation MAE per Fold (Random Forest)')
plt.xlabel('Fold')
plt.ylabel('MAE')
plt.grid(True)
plt.show()


In [ ]:
#tuning
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 8]
}

random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=15,
    cv=5,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)
print("Best Params:", random_search.best_params_)


In [ ]:
# Step 1: Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

# Step 2: Prepare Data
# Replace df_final with your actual DataFrame if named differently
X = df_final.drop(columns=['CLAIM_AMOUNT'])
y = df_final['CLAIM_AMOUNT']

# Step 3: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Train the Random Forest with Best Parameters
best_rf_tuned = RandomForestRegressor(
    n_estimators=300,
    max_depth=30,
    min_samples_split=5,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1
)

best_rf_tuned.fit(X_train, y_train)

# Step 5: Predict
y_train_pred_tuned = best_rf_tuned.predict(X_train)
y_test_pred_tuned = best_rf_tuned.predict(X_test)

# Step 6: Evaluate
print("🔹 Train Performance (Tuned Model):")
print(f"   MAE: {mean_absolute_error(y_train, y_train_pred_tuned):.2f}")
print(f"   RMSE: {mean_squared_error(y_train, y_train_pred_tuned)**0.5:.2f}")
print(f"   R²: {r2_score(y_train, y_train_pred_tuned):.4f}")

print("\n🔹 Test Performance (Tuned Model):")
print(f"   MAE: {mean_absolute_error(y_test, y_test_pred_tuned):.2f}")
print(f"   RMSE: {mean_squared_error(y_test, y_test_pred_tuned)**0.5:.2f}")
print(f"   R²: {r2_score(y_test, y_test_pred_tuned):.4f}")

# Step 7: Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_mae_scores_tuned = cross_val_score(best_rf_tuned, X, y, cv=kf, scoring='neg_mean_absolute_error')
mae_scores_tuned = -cv_mae_scores_tuned

print("\n🔸 Cross-Validation (5-Fold) MAE (Tuned Model):")
print(f"Scores: {mae_scores_tuned}")
print(f"Average MAE: {mae_scores_tuned.mean():.2f}")

# Step 8: Plot Cross-Validation MAE per Fold
plt.figure(figsize=(8,5))
plt.plot(range(1, 6), mae_scores_tuned, marker='o', color='blue')
plt.title('Cross-Validation MAE per Fold (Random Forest - Tuned)')
plt.xlabel('Fold')
plt.ylabel('MAE')
plt.grid(True)
plt.show()

# Step 9: Feature Importance Visualization
importances = best_rf_tuned.feature_importances_
feature_names = X_train.columns
sorted_indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.bar(range(10), importances[sorted_indices][:10], align='center', color='green')
plt.xticks(range(10), feature_names[sorted_indices][:10], rotation=45)
plt.xlabel("Feature Importance")
plt.ylabel("Relative Importance")
plt.title("Top 10 Important Features (Tuned Model)")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

xgb.plot_importance(xgb_model_sel, max_num_features=16, height=0.5)
plt.title("Feature Importance (Top 16)")
plt.tight_layout()
plt.show()


In [ ]:
import shap

# Create SHAP explainer
explainer = shap.Explainer(xgb_model_sel)
shap_values = explainer(X_test_sel)

# Summary plot
shap.summary_plot(shap_values, features=X_test_sel, feature_names=X.columns[selector.get_support()])


In [ ]:
import pandas as pd

# Create a DataFrame for model comparison
data = {
    "Metric": [
        "Train MAE", "Train RMSE", "Train R²",
        "Test MAE", "Test RMSE", "Test R²",
        
    ],
    "Random Forest (Tuned)": [
        94.04, 1275.42, 0.9322,
        98.07, 859.65, 0.9529,
       
    ],
    "XGBoost (Tuned)": [
        17.97, 65.07, 0.9998,
        89.89, 538.72, 0.9815,
        
    ]
}

df_comparison = pd.DataFrame(data)

# Print the table
print(df_comparison.to_string(index=False))
